# Antasena Super Course — Perbandingan Efisiensi (Simulasi)

**Tujuan:** membandingkan dua data telemetri dan mencari tahu *mengapa* salah satunya lebih hemat energi.

Setiap kali kamu menemukan `____` di sel kode, ganti dengan kode yang benar lalu jalankan selnya.

Jalankan sel dari atas ke bawah, karena sel di bawah bergantung pada sel sebelumnya. Jika muncul `NameError: name '____' is not defined`, berarti masih ada bagian yang belum diisi.

## Persiapan (khusus Google Colab)

Jika kamu membuka notebook ini di **Google Colab**, jalankan sel di bawah terlebih dahulu. Sel ini mengunduh kedua file data CSV dari folder `data/` di repo kursus. Di Jupyter lokal, sel ini tidak melakukan apa-apa.

In [1]:
import os, sys, urllib.request

if "google.colab" in sys.modules:
    RAW = "https://raw.githubusercontent.com/Antasena-ITS-Team/ASC_Simulator/master/data/"
    os.makedirs("data", exist_ok=True)
    for nama in ["DataASC_105V_9A.csv", "DataASC_80V_12A.csv"]:
        if not os.path.exists(f"data/{nama}"):
            urllib.request.urlretrieve(RAW + nama, f"data/{nama}")
    print("Data siap:", sorted(os.listdir("data")))

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Gaya grafik yang bersih: tanpa bingkai atas/kanan, grid tipis
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.titleweight": "bold",
})

## Konfigurasi

Setiap baris berisi `(path_csv, label)`.

In [3]:
ATTEMPTS = [
    ("data/DataASC_105V_9A.csv", "105V / 9A"),
    ("data/DataASC_80V_12A.csv", "80V / 12A"),
]

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # aman untuk buta warna dengan urutan ini
COLORS = {label: PALETTE[i % len(PALETTE)] for i, (_, label) in enumerate(ATTEMPTS)}

BASE, OTHER = ATTEMPTS[0][1], ATTEMPTS[1][1]  # percobaan acuan dan pembanding

## Langkah 1 — Memuat data dan menghitung besaran turunan

- `speed_ms` — jika file menyimpan `speed_kmh`, ubah satuannya (**1 m/s = 3,6 km/jam**)
- `power_W = voltage_V × current_A` (daya = tegangan × arus)
- `dt_s` — selang waktu antar baris (`.diff()`)
- `energy_Wh` — jumlah kumulatif `power_W × dt_s`, diubah dari joule (W·s) ke Wh (**1 Wh = 3600 J**)
- `efficiency_km_per_kWh = distance_m / energy_Wh` — kebetulan m/Wh dan km/kWh bernilai sama

**Cek kewajaran:** *nama* kolom belum tentu menjamin *satuannya*. `check_speed_units` membandingkan kolom kecepatan dengan kecepatan yang dihitung dari `distance_m` dan `time_s`. Jika keduanya berbeda 3,6 kali, berarti kolom itu sebenarnya dalam km/jam.


In [ ]:
def check_speed_units(df, label):
    implied = df["distance_m"].diff() / df["time_s"].diff()
    moving = df["speed_ms"] > 1
    ratio = (df.loc[moving, "speed_ms"] / implied[moving]).median()
    if abs(ratio - 3.6) < 0.1:
        print(f"[{label}] speed_ms bernilai {ratio:.2f}x kecepatan dari jarak/waktu -> sebenarnya km/jam, dikonversi")
        df["speed_ms"] = df["speed_ms"] / 3.6
    return df


def load_attempt(path, label):
    df = pd.read_csv(path)

    if "speed_kmh" in df.columns:
        df["speed_ms"] = df["speed_kmh"] / 3.6
    df = check_speed_units(df, label)
# ubah bagian bawah ini saja 
    df["power_W"] = df["___"] * df["____"]
    df["dt_s"] = df["time_s"].diff().fillna(0)
    df["energy_Wh"] = (df["___"] * df["___"] / ____).cumsum()
    df["efficiency_km_per_kWh"] = np.where(
        #Hitung Efisiensi
        df["energy_Wh"] > 0, df["___"] / df["____"], np.nan
    )
    df["attempt"] = label
    return df
# sampai sini

data = {label: load_attempt(path, label) for path, label in ATTEMPTS}

for label, df in data.items():
    print(f"--- {label} ---")
    display(df.head(3))

## Langkah 2 — Sinyal mentah

Aturan membaca semua grafik di notebook ini: **kedua percobaan selalu digambar bersama**, dan garis putus-putus menunjukkan **rata-rata** masing-masing percobaan (nilainya tertulis di legenda).

- Tegangan dan kecepatan cukup stabil, jadi digambar **terhadap waktu**. Perhatikan bahwa keduanya finis hampir bersamaan (~35 menit), tetapi **pola kecepatannya** sangat berbeda.
- `plot_over_time(column, ylabel, title)` menggambar satu garis untuk setiap percobaan.



In [ ]:
def plot_over_time(column, ylabel, title):
    plt.figure(figsize=(10, 4.5))
    for label, df in data.items():
        avg = df[column].mean()
        #plot terhadap waktu (x , y)
        plt.plot(df[""], df[column], color=COLORS[label], linewidth=1.2,
        ##
                 label=f"{label} (rata-rata {avg:.1f})")
        plt.axhline(avg, color=COLORS[label], linestyle="--", linewidth=1)
    plt.xlabel("waktu (s)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

#test fungsi diatas
plot_over_time("___", "____", "____")

NameError: name 'data' is not defined

<Figure size 1000x450 with 0 Axes>

In [ ]:
plot_over_time("speed_ms", "kecepatan (m/s)", "Kecepatan terhadap waktu")

### Satu lap, posisi demi posisi

Arus dan daya naik-turun sangat cepat, jadi jika digambar terhadap waktu, kedua garis akan bertumpuk dan sulit dibaca. Selain itu, pada detik yang sama kedua mobil berada di **tempat yang berbeda** di lintasan.

Solusinya: ambil **satu lap** dan gambar terhadap **posisi dalam lap (m)**. Tanjakan dan turunan selalu berada di posisi yang sama, jadi kedua percobaan bisa dibandingkan titik demi titik. **Arsiran abu-abu = selisih antara kedua percobaan.**


In [ ]:
LAP = __                            # input lap 1 atau 2 atau 3
POS = np.arange(0, 3600, 5)        


def plot_one_lap(column, ylabel, title):
    y = {}
    plt.figure(figsize=(10, 4.5))
    for label, df in data.items():
        one = df[df["lap"] == LAP]
        pos = one["distance_m"] - one["distance_m"].iloc[0]   # jarak dari garis start lap
        y[label] = np.interp(POS, pos, one[column])            
        avg = one[column].mean()
        plt.plot(POS, y[label], color=COLORS[label], linewidth=1.5, label=f"{label} (rata-rata {avg:.1f})")
        plt.axhline(avg, color=COLORS[label], linestyle="--", linewidth=1)
    plt.fill_between(POS, y[BASE], y[OTHER], color="#999", alpha=0.3, linewidth=0, label="selisih")
    plt.ylim(0, max(v.max() for v in y.values()) * 1.25) 
    plt.xlabel(f"posisi dalam lap {LAP} (m)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(frameon=False, ncol=3, loc="upper left")
    plt.tight_layout()
    plt.show()

# input data arus
plot_one_lap("____", "____", f"Arus sepanjang lap {LAP}")

NameError: name 'data' is not defined

<Figure size 1000x450 with 0 Axes>

## Langkah 3 — Daya dan energi

Pada grafik energi terhadap jarak, **garis yang lebih landai = jarak per Wh lebih jauh = lebih efisien.** Arsiran abu-abu adalah energi tambahan yang dipakai percobaan yang lebih boros.

`plot_over_distance(column, ylabel, title, unit)` bekerja seperti `plot_over_time`, tetapi sumbu x-nya **jarak**. Di dalamnya, `np.interp` membaca nilai kedua percobaan di titik jarak yang **sama** (setiap 10 m), supaya selisihnya bisa diarsir. Nilai akhir setiap percobaan ditulis di ujung kanan garis.


In [ ]:
plot_one_lap(____, "daya (W)", f"Daya sepanjang lap {LAP}")

In [ ]:
def plot_over_distance(column, ylabel, title, unit="", ylim=None, belum_stabil_m=0):
    jarak = np.arange(0, min(df["distance_m"].max() for df in data.values()), 10)
    y = {}
    plt.figure(figsize=(10, 4.5))
    for label, df in data.items():
        y[label] = np.interp(jarak, df["distance_m"], df[column])   
        plt.plot(jarak, y[label], color=COLORS[label], linewidth=2, label=label)
        plt.text(jarak[-1], y[label][-1], f"  {y[label][-1]:.1f} {unit}", color=COLORS[label],
                 va="center", fontweight="bold")
    plt.fill_between(jarak, y[BASE], y[OTHER], color="#999", alpha=0.3, linewidth=0, label="selisih")
    if belum_stabil_m:                                                  
        plt.axvspan(0, belum_stabil_m, color="#999", alpha=0.15, linewidth=0)
        plt.text(belum_stabil_m / 2, ylim[0] + 2, "belum stabil", ha="center", color="#666")
    if ylim:
        plt.ylim(*ylim)
    plt.xlim(right=jarak[-1] * 1.1)                                     
    plt.xlabel("jarak (m)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()
    return y

## input
energi = plot_over_distance("energy_Wh", "energi kumulatif (Wh)",
                            "Energi yang dipakai per jarak tempuh (lebih landai = lebih efisien)", unit="Wh")
print(f"Selisih energi di akhir: {energi[OTHER][-1] - energi[BASE][-1]:.1f} Wh")

## Langkah 4 — Perbandingan efisiensi

Efisiensi kumulatif masih naik-turun di awal (karena dibagi energi yang masih sangat kecil, bagian berarsir) lalu stabil seiring berjalannya balapan. Bandingkan nilai **akhirnya**, yang tertulis di ujung kanan setiap garis.



In [ ]:
#  input data efisiensi dan labelnya
plot_over_distance("______", "efisiensi (km/kWh)", "Efisiensi kumulatif sepanjang balapan",
                   ylim=(40, 80), belum_stabil_m=1000)
##
final = {}
for label, df in data.items():
    final[label] = df["efficiency_km_per_kWh"].iloc[-1]
    print(f"{label}: efisiensi akhir = {final[label]:.2f} km/kWh")
print(f"{BASE} {(final[BASE] / final[OTHER] - 1) * 100:+.0f}% dibanding {OTHER}")

NameError: name 'plot_over_distance' is not defined

### Ringkasan per lap

Untuk setiap lap: jarak tempuh = `distance_m` terakhir − pertama; begitu juga untuk `energy_Wh`.

Ringkas `lap_distance_m`, `lap_energy_Wh`, dan `lap_efficiency_km_per_kWh`.

In [ ]:
summaries = []
for label, df in data.items():
    g = df.groupby("lap").agg(
        distance_start=("distance_m", "first"), distance_end=("distance_m", "last"),
        energy_start=("energy_Wh", "first"), energy_end=("energy_Wh", "last"),
        avg_speed_ms=("speed_ms", "mean"), max_speed_ms=("speed_ms", "max"),
        avg_current_A=("current_A", "mean"),
    )
    #input summary
    g["lap_distance_m"] = g["distance_end"] - g["distance_start"]
    g["lap_energy_Wh"] = g["energy_end"] - g["energy_start"]
    g["lap_efficiency_km_per_kWh"] = g["_____"] / g["______"]
    g["attempt"] = label
    ###
    summaries.append(g.reset_index()[[
        "attempt", "lap", "lap_distance_m", "lap_energy_Wh",
        "lap_efficiency_km_per_kWh", "avg_speed_ms", "max_speed_ms", "avg_current_A",
    ]])

lap_summary = pd.concat(summaries, ignore_index=True)
lap_summary.round(2)

In [ ]:
eff_lap = lap_summary.pivot(index="lap", columns="attempt", values="lap_efficiency_km_per_kWh")

plt.figure(figsize=(8, 4.5))
for label in data:
    plt.plot(eff_lap.index, eff_lap[label], color=COLORS[label], marker="o", linewidth=2, label=label)
plt.fill_between(eff_lap.index, eff_lap[BASE], eff_lap[OTHER], color="#999", alpha=0.3, linewidth=0, label="selisih")
for lap in eff_lap.index:
    beda = (eff_lap.loc[lap, BASE] / eff_lap.loc[lap, OTHER] - 1) * 100
    plt.text(lap, eff_lap.loc[lap].max() + 3, f"{beda:+.0f}%", ha="center", va="bottom", fontweight="bold")
plt.xticks(eff_lap.index)
plt.ylim(0, eff_lap.max().max() * 1.2)
plt.xlabel("lap")
plt.ylabel("efisiensi (km/kWh)")
plt.title(f"Efisiensi per lap (angka = {BASE} dibanding {OTHER})")
plt.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

## Langkah 5 — Mengapa efisiensinya berbeda?

Tabel di bawah mengumpulkan angka-angka yang kamu perlukan untuk menjawab pertanyaan.

In [ ]:
overall = pd.DataFrame({
    label: {
        "efisiensi akhir (km/kWh)": df["efficiency_km_per_kWh"].iloc[-1],
        "total energi (Wh)": df["energy_Wh"].iloc[-1],
        "rata-rata tegangan (V)": df["voltage_V"].mean(),
        "rata-rata arus (A)": df["current_A"].mean(),
        "rata-rata arus² (A²)": (df["current_A"] ** 2).mean(),
        "rata-rata kecepatan (m/s)": df["speed_ms"].mean(),
        "kecepatan maks (m/s)": df["speed_ms"].max(),
        "waktu tempuh (s)": df["time_s"].iloc[-1],
    }
    for label, df in data.items()
})
overall.round(2)

## Apa yang bisa kamu rekomendasikan?
